# Notebook 02: Recommendation System Design

## Why This Matters

Recommendation systems drive a disproportionate share of engagement at every major tech platform: 70% of YouTube watch time, 35% of Amazon purchases, and the core product loop at Netflix, Spotify, and TikTok. Designing a recommendation system from scratch is one of the most common ML system design interview questions, and getting the two-stage architecture right is essential. A naive single-model approach breaks at billion-scale - a recommender that can't return results in <100ms is useless, so the architecture must decouple the 'find candidates' problem from the 'precisely rank candidates' problem.

## Background

The fundamental challenge is the **item selection problem**: given a catalog of millions to billions of items and a user, which items should we show? This is computationally intractable to solve exactly in real-time, so all production recommendation systems use a two-stage pipeline: fast approximate retrieval followed by precise reranking.

In [ ]:
%pip install -q torch numpy pandas scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
torch.manual_seed(42)
np.random.seed(42)
print("Ready.")

## 1. Two-Stage Architecture: Why Not One Model?

The naive approach - score every item in the catalog for every user - doesn't scale. At Netflix scale (240M users x 15,000 titles), that's 3.6 billion forward passes per recommendation request. Even at 1 microsecond per inference, that's 3,600 seconds. Unacceptable.

**Stage 1 - Retrieval (Candidate Generation)**:
- Goal: reduce 10M+ items to ~100-1000 candidates in <10ms
- Method: fast approximate nearest neighbor (ANN) search in embedding space
- Recall matters more than precision here

**Stage 2 - Ranking**:
- Goal: precisely score the ~100-1000 candidates from Stage 1
- Method: rich feature model with user history, item features, cross features
- Precision matters here - this determines what the user actually sees

**Key insight**: Stage 1 maximizes recall (don't miss good items). Stage 2 maximizes precision (rank the truly best items at the top). Conflating these is a design error.

In [ ]:
N_USERS = 200
N_ITEMS = 500
N_INTERACTIONS = 5000

users = 200_000_000
items_full = 10_000_000
items_retrieved = 500
inference_us = 1

full_catalog_ms = (users * items_full * inference_us) / 1e9
print("=== Scale Analysis ===")
print(f"Full catalog scoring time: {full_catalog_ms:,.0f}s - IMPOSSIBLE in real-time")
print(f"Speedup with two-stage: {items_full / items_retrieved:,}x")
print(f"Per-user retrieval: ~{items_retrieved * inference_us / 1000:.3f}ms - feasible!")
print(f"\nThe two-stage architecture is not optional at scale - it is required.")

## 2. Two-Tower Retrieval Model

The two-tower (bi-encoder) model learns embeddings for users and items in a shared vector space. Similar items and users end up close together.

**User tower inputs**: user ID embedding + historical item embeddings + user features -> MLP -> user embedding
**Item tower inputs**: item ID embedding + item features -> MLP -> item embedding
**Training objective**: maximize dot product for (user, interacted item) pairs vs random negatives

**Why shared embedding space matters**: Both towers must produce vectors in the same space so that dot product similarity is meaningful. This is enforced by training them jointly with the same loss.

In [ ]:
class TwoTowerModel(nn.Module):
    def __init__(self, n_users, n_items, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.user_mlp = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, embed_dim))
        self.item_embed = nn.Embedding(n_items, embed_dim)
        self.item_mlp = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, embed_dim))
    
    def user_forward(self, user_ids):
        u = self.user_embed(user_ids)
        return F.normalize(self.user_mlp(u), dim=-1)
    
    def item_forward(self, item_ids):
        v = self.item_embed(item_ids)
        return F.normalize(self.item_mlp(v), dim=-1)
    
    def forward(self, user_ids, pos_item_ids, neg_item_ids):
        return self.user_forward(user_ids), self.item_forward(pos_item_ids), self.item_forward(neg_item_ids)

def bpr_loss(u, pos_v, neg_v, temperature=0.1):
    # BPR: maximize positive score over negative score
    pos_score = (u * pos_v).sum(-1) / temperature
    neg_score = (u * neg_v).sum(-1) / temperature
    return -F.logsigmoid(pos_score - neg_score).mean()

N_USERS_SIM, N_ITEMS_SIM, EMBED_DIM = 200, 500, 32
N_INTERACTIONS_SIM = 5000
user_ids = torch.randint(0, N_USERS_SIM, (N_INTERACTIONS_SIM,))
pos_item_ids = torch.randint(0, N_ITEMS_SIM, (N_INTERACTIONS_SIM,))
neg_item_ids = torch.randint(0, N_ITEMS_SIM, (N_INTERACTIONS_SIM,))

model = TwoTowerModel(N_USERS_SIM, N_ITEMS_SIM, embed_dim=EMBED_DIM)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
losses = []
for epoch in range(30):
    model.train()
    epoch_loss = 0
    for i in range(0, N_INTERACTIONS_SIM, 256):
        u_b = user_ids[i:i+256]; p_b = pos_item_ids[i:i+256]; n_b = neg_item_ids[i:i+256]
        optimizer.zero_grad()
        u, pv, nv = model(u_b, p_b, n_b)
        loss = bpr_loss(u, pv, nv)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / (N_INTERACTIONS_SIM // 256))

import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(losses)
plt.title("Two-Tower Training Loss (BPR)")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.tight_layout(); plt.savefig("/tmp/two_tower_loss.png", dpi=80); plt.show()
print(f"Final BPR loss: {losses[-1]:.4f}")
print(f"User embeddings: {model.user_forward(torch.arange(N_USERS_SIM)).shape}")
print(f"Item embeddings: {model.item_forward(torch.arange(N_ITEMS_SIM)).shape}")

## 3. Evaluation Metrics

- **NDCG@K**: Normalized Discounted Cumulative Gain - measures ranking quality with logarithmic discount
- **MRR**: Mean Reciprocal Rank - average 1/rank of first relevant item
- **Precision@K**: fraction of top-K recommendations that are relevant
- **Recall@K**: fraction of all relevant items captured in top-K
- **Diversity**: average pairwise dissimilarity - prevents filter bubbles
- **Novelty**: fraction of recommendations the user hasn't seen before

In [ ]:
import numpy as np

def ndcg_at_k(relevance, k):
    rel = np.array(relevance[:k], dtype=float)
    gains = rel / np.log2(np.arange(2, len(rel) + 2))
    dcg = gains.sum()
    ideal = np.sort(rel)[::-1]
    idcg = (ideal / np.log2(np.arange(2, len(ideal) + 2))).sum()
    return dcg / idcg if idcg > 0 else 0.0

def mrr(relevant_ranked):
    for i, r in enumerate(relevant_ranked):
        if r == 1: return 1.0 / (i + 1)
    return 0.0

def precision_at_k(recommended, relevant_set, k):
    return sum(1 for item in recommended[:k] if item in relevant_set) / k

def recall_at_k(recommended, relevant_set, k):
    hits = sum(1 for item in recommended[:k] if item in relevant_set)
    return hits / len(relevant_set) if relevant_set else 0.0

recommended = [3, 1, 7, 2, 9, 5, 8, 4, 6, 0]
relevant_set = {1, 3, 7}
relevance_scores = [1 if r in relevant_set else 0 for r in recommended]
K = 5
print(f"=== Recommendation Metrics (K={K}) ===")
print(f"Recommended: {recommended[:K]}")
print(f"Relevant:    {sorted(relevant_set)}")
print(f"NDCG@{K}:      {ndcg_at_k(relevance_scores, K):.4f}")
print(f"MRR:          {mrr(relevance_scores):.4f}")
print(f"Precision@{K}: {precision_at_k(recommended, relevant_set, K):.4f}")
print(f"Recall@{K}:    {recall_at_k(recommended, relevant_set, K):.4f}")

## 4. Cold Start Problem

Cold start occurs in three forms:

1. **New user**: no interaction history -> no user embedding -> can't retrieve personalized items
2. **New item**: no user interactions -> item embedding not learned -> item never retrieved
3. **New system**: both problems at once

**Solutions**:
- **New users**: use content-based features, prompt for explicit preferences, use popular items as fallback
- **New items**: use item content features (text embedding, metadata), build a content-based tower
- **Hybrid**: blend collaborative filtering with content-based; weight toward content early, toward CF as data accumulates

**Interview tip**: Always proactively ask 'How do we handle new users and new items?' Interviewers expect this.

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Two-stage architecture | Retrieval (recall) -> Ranking (precision); required at scale |
| Two-tower model | User + item embeddings in shared space; ANN search at inference |
| BPR loss | Maximize score of positive over negative; standard for implicit feedback |
| NDCG@K | Measures ranking quality with position discount; primary offline metric |
| Cold start | New user/item has no interaction data; need content-based fallback |

### Common Pitfalls
- Designing a single model to do both retrieval and ranking
- Training only on positive interactions without negative sampling
- Evaluating only offline without A/B testing online
- Ignoring cold start in the design
